# NB09: Mine Proximity Features and Extended Regional Analysis

Addresses three REPORT.md limitations:
- **Limitation 1**: No model beats B0 for Cu in spatial block CV.
- **Limitation 4**: H8 limited to 2 usable regions (binary class-balance constraint).
- **Limitation 5**: Pb threshold discrimination poor (sensitivity=0.25 for >100ppm).

**Approach**: Join mine proximity features (mine/TRI/NPL distance) from
`metal_contamination_response/data/00_site_classification.csv` to the feature matrix
via haversine nearest-neighbor (10 km threshold). Add a binary `has_mine_prox_data`
indicator so XGBoost can distinguish sentinel-imputed uncovered samples from
genuinely far-from-mine samples. Re-frame H8 as continuous log_mine_prox_km
regression to enable all 8 k-means regions (not just 2).

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, roc_auc_score, roc_curve

sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, PALETTE, FIGW, ROW_H, grid_h
apply_style()

# Path setup — works from notebooks/ or project root
BASE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = BASE / 'data'
FIGS = BASE / 'figures'
MCR  = BASE.parent / 'metal_contamination_response' / 'data'

# Add scripts dir
for _cand in [BASE / 'scripts', Path.cwd() / 'scripts']:
    if _cand.exists():
        sys.path.insert(0, str(_cand))
        break

fm = pd.read_parquet(DATA / 'feature_matrix.parquet')
sc = pd.read_csv(MCR / '00_site_classification.csv')
print(f'Feature matrix: {fm.shape}')
print(f'Site classification: {sc.shape}')
print(f'SC columns: {list(sc.columns)}')

Feature matrix: (42037, 354)
Site classification: (278952, 7)
SC columns: ['sample_id', 'site_class', 'lat', 'lon', 'mine_proximity_km', 'tri_proximity_km', 'npl_proximity_km']


## Section A: Haversine nearest-neighbor mine proximity join

Use `scipy.spatial.cKDTree` on lat/lon coordinates to find the nearest
`site_classification.csv` point for each feature_matrix sample. Inherit
mine_proximity_km if within 10 km. Add sentinel imputation and a binary
`has_mine_prox_data` column so XGBoost can distinguish 'no data' from
'genuinely remote'.

In [2]:
# Filter site_classification to rows with mine_proximity_km and unique lat/lon
sc_valid = sc[sc['mine_proximity_km'].notna()].copy()
sc_valid = sc_valid.drop_duplicates(subset=['lat', 'lon'])
print(f'Unique SC coords with mine_proximity_km: {len(sc_valid):,}')

# Build KD-tree in lat/lon space
tree = cKDTree(sc_valid[['lat', 'lon']].values)
fm_coords = fm[['lat', 'lon']].values
_dists_deg, idxs = tree.query(fm_coords, k=1)

# Haversine distance in km
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (np.sin(dlat / 2) ** 2
         + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon / 2) ** 2)
    return 2 * R * np.arcsin(np.sqrt(a))

matched = sc_valid.iloc[idxs].reset_index(drop=True)
dists_km = haversine_km(
    fm['lat'].values, fm['lon'].values,
    matched['lat'].values, matched['lon'].values,
)

THRESHOLD_KM = 10.0
mask = dists_km <= THRESHOLD_KM
n_covered = mask.sum()
pct = n_covered / len(fm) * 100
print(f'Covered within {THRESHOLD_KM} km: {n_covered:,} ({pct:.1f}%)')
print(f'Uncovered: {(~mask).sum():,}')

if n_covered < 1000:
    print('WARNING: coverage below 1,000 samples — mine proximity features will have'
          ' limited predictive signal; interpret M2_mine results cautiously.')

# Build extended feature matrix
fm_ext = fm.copy()

# Binary indicator: 1 if within threshold, 0 otherwise
# Placed FIRST so XGBoost sees it alongside the sentinel-imputed continuous features
fm_ext['has_mine_prox_data'] = mask.astype(np.float32)

# Inherit proximity values for covered samples
fm_ext['mine_proximity_km'] = np.where(mask, matched['mine_proximity_km'].values, np.nan)
fm_ext['tri_proximity_km']  = np.where(mask, matched['tri_proximity_km'].values, np.nan)
fm_ext['npl_proximity_km']  = np.where(mask, matched['npl_proximity_km'].values, np.nan)

# Log-transform + sentinel imputation for uncovered rows
# Sentinel = log1p(9999) ≈ 9.21; distinguishable from real data by has_mine_prox_data=0
SENTINEL = np.log1p(9999.0)
fm_ext['log_mine_prox_km'] = np.where(
    mask, np.log1p(fm_ext['mine_proximity_km']), SENTINEL)
fm_ext['log_tri_prox_km'] = np.where(
    mask, np.log1p(fm_ext['tri_proximity_km'].fillna(9999)), SENTINEL)
fm_ext['log_npl_prox_km'] = np.where(
    mask, np.log1p(fm_ext['npl_proximity_km'].fillna(9999)), SENTINEL)

print(f'\nlog_mine_prox_km stats (covered, n={n_covered}):')
print(fm_ext.loc[mask, 'log_mine_prox_km'].describe().round(3))
print(f'Sentinel value: {SENTINEL:.3f}')

Unique SC coords with mine_proximity_km: 8,672
Covered within 10.0 km: 15,476 (36.8%)
Uncovered: 26,561

log_mine_prox_km stats (covered, n=15476):
count    15476.000
mean         1.304
std          0.714
min          0.068
25%          0.631
50%          1.212
75%          1.623
max          3.038
Name: log_mine_prox_km, dtype: float64
Sentinel value: 9.210


## Section B: M2_mine spatial block CV (Limitations 1 & 5)

Run leave-one-spatial-block-out CV for M2_mine (CLR + GW + env + mine proximity).
Compare RMSE against B0 and M2 baselines. Use OOF predictions for threshold
discrimination analysis (Cu/Pb at 100 ppm).

In [3]:
from modelling import run_spatial_block_cv, TARGETS

spatial_blocks = pd.read_csv(DATA / 'spatial_blocks.csv', index_col='sample_id')
blocks_series = spatial_blocks['block'].reindex(fm_ext.index)
print(f'Blocks aligned: {blocks_series.notna().sum():,} of {len(fm_ext):,} samples')
print(f'Block distribution:\n{blocks_series.value_counts().sort_index()}')

Blocks aligned: 42,037 of 42,037 samples
Block distribution:
block
0    15630
1    11649
2     1180
3     5204
4     8374
Name: count, dtype: int64


In [4]:
mine_cv_rows = []
oof_preds = {}  # target → OOF Series

for target in TARGETS:
    results_df, oof = run_spatial_block_cv(
        fm_ext, target, 'M2_mine', blocks_series,
        model_type='xgboost', return_oof=True,
    )
    overall_rmse = results_df['rmse'].mean()
    mine_cv_rows.append({'target': target, 'model': 'M2_mine', 'rmse': overall_rmse})
    oof_preds[target] = oof
    print(f'{target}: M2_mine RMSE = {overall_rmse:.4f} '
          f'(n_blocks={len(results_df)}, '
          f'n_train_min={results_df["n_train"].min()}, '
          f'n_test_min={results_df["n_test"].min()})')

mine_cv_df = pd.DataFrame(mine_cv_rows).set_index('target')

# Load M2 and B0 baselines (per-block rows → mean RMSE)
baseline_cv = pd.read_csv(DATA / 'cv_results_models.csv')
m2_rmse = baseline_cv[baseline_cv['model'] == 'M2'].groupby('target')['rmse'].mean()
b0_rmse = pd.read_csv(DATA / 'cv_results_baselines.csv')
b0_rmse = b0_rmse[b0_rmse['model'] == 'B0'].groupby('target')['rmse'].mean()

print('\n--- RMSE comparison ---')
for t in TARGETS:
    mine = mine_cv_df.loc[t, 'rmse']
    m2   = m2_rmse.get(t, float('nan'))
    b0   = b0_rmse.get(t, float('nan'))
    print(f'{t}: B0={b0:.4f}  M2={m2:.4f}  M2_mine={mine:.4f}  '
          f'Δ(M2_mine−M2)={mine-m2:+.4f}  beats_B0={mine < b0}')

mine_cv_df.reset_index().to_csv(DATA / 'cv_results_m2_mine.csv', index=False)

log_Cu_ppm: M2_mine RMSE = 1.1991 (n_blocks=5, n_train_min=13794, n_test_min=414)


log_Zn_ppm: M2_mine RMSE = 0.9305 (n_blocks=5, n_train_min=14488, n_test_min=406)


log_Pb_ppm: M2_mine RMSE = 0.7734 (n_blocks=5, n_train_min=14398, n_test_min=492)


log_Ni_ppm: M2_mine RMSE = 1.8223 (n_blocks=5, n_train_min=15242, n_test_min=543)

--- RMSE comparison ---
log_Cu_ppm: B0=1.1146  M2=1.1210  M2_mine=1.1991  Δ(M2_mine−M2)=+0.0781  beats_B0=False
log_Zn_ppm: B0=0.6781  M2=0.9229  M2_mine=0.9305  Δ(M2_mine−M2)=+0.0076  beats_B0=False
log_Pb_ppm: B0=0.8802  M2=0.8260  M2_mine=0.7734  Δ(M2_mine−M2)=-0.0526  beats_B0=True
log_Ni_ppm: B0=1.7018  M2=1.8640  M2_mine=1.8223  Δ(M2_mine−M2)=-0.0417  beats_B0=False


In [5]:
# Threshold discrimination (Limitation 5): Cu and Pb at 100 ppm
print('--- Threshold discrimination: M2_mine OOF predictions ---')
thresh_rows = []
for target, metal_ppm in [('log_Cu_ppm', 100), ('log_Pb_ppm', 100)]:
    log_thresh = np.log1p(metal_ppm)
    y_true = fm_ext[target]
    y_pred = oof_preds[target]
    valid  = y_true.notna() & y_pred.notna()
    y_tv   = y_true[valid]
    y_pv   = y_pred[valid]
    pos    = (y_tv >= log_thresh).astype(int)
    n_pos  = pos.sum()
    if n_pos < 5:
        print(f'{target}: too few positives ({n_pos}) — skip')
        continue
    auc = roc_auc_score(pos, y_pv)
    fpr, tpr, _ = roc_curve(pos, y_pv)
    youden_idx = np.argmax(tpr - fpr)
    sens = tpr[youden_idx]
    spec = 1 - fpr[youden_idx]
    print(f'{target} >{metal_ppm}ppm: n_pos={n_pos}, AUC={auc:.3f}, '
          f'sensitivity={sens:.3f}, specificity={spec:.3f}')
    thresh_rows.append({'target': target, 'threshold_ppm': metal_ppm,
                        'n_pos': n_pos, 'auc': auc,
                        'sensitivity': sens, 'specificity': spec})

pd.DataFrame(thresh_rows).to_csv(DATA / 'threshold_disc_m2_mine.csv', index=False)

--- Threshold discrimination: M2_mine OOF predictions ---
log_Cu_ppm >100ppm: n_pos=2401, AUC=0.638, sensitivity=0.786, specificity=0.493
log_Pb_ppm >100ppm: n_pos=40, AUC=0.620, sensitivity=1.000, specificity=0.570


## Section C: Extended H8 — continuous mine proximity regression (Limitation 4)

Re-frames H8 from binary contamination classification to continuous
log_mine_prox_km regression. Binary framing limited analysis to 2 of 8 k-means
regions (class balance requirement). Continuous framing allows any region with
variation in mine_proximity_km to contribute.

**H8 extended criterion**: ≥4 of 8 regions have CLR within-region R² > 0.05, AND
CLR+GW adds ≥0.01 mean R² over CLR.

In [6]:
# Filter to samples with real mine proximity data (not sentinel-imputed)
mine_mask = fm_ext['has_mine_prox_data'] == 1
fm_mine = fm_ext[mine_mask].copy()
print(f'Samples with mine proximity coverage: {len(fm_mine):,}')
print(f'log_mine_prox_km range: '
      f'{fm_mine["log_mine_prox_km"].min():.2f} – '
      f'{fm_mine["log_mine_prox_km"].max():.2f}')
print(f'log_mine_prox_km std: {fm_mine["log_mine_prox_km"].std():.3f}')

if len(fm_mine) < 1000:
    print('WARNING: fewer than 1,000 covered samples — H8 extended regression '
          'will be underpowered.')

# k-means on lat/lon (k=8, matching NB07)
km = KMeans(n_clusters=8, random_state=42, n_init=10)
fm_mine = fm_mine.copy()
fm_mine['region'] = km.fit_predict(fm_mine[['lat', 'lon']].values)

print('\nRegion sizes:')
print(fm_mine['region'].value_counts().sort_index().to_frame('n'))

Samples with mine proximity coverage: 15,476
log_mine_prox_km range: 0.07 – 3.04
log_mine_prox_km std: 0.714

Region sizes:
           n
region      
0       1465
1       9822
2        237
3       2108
4        142
5        174
6       1138
7        390


In [7]:
CLR_COLS = [c for c in fm_mine.columns if c.startswith('clr_')]
GW_COLS  = [c for c in fm_mine.columns if c.startswith('gw_')]

TARGET_CONT = 'log_mine_prox_km'
MIN_N = 20

within_rows = []
for region in sorted(fm_mine['region'].unique()):
    sub = fm_mine[fm_mine['region'] == region]
    if len(sub) < MIN_N:
        print(f'Region {region}: skipped (n={len(sub)})')
        continue
    y_reg = sub[TARGET_CONT].values
    y_var = np.var(y_reg)
    if y_var < 1e-6:
        print(f'Region {region}: skipped (zero variance in log_mine_prox_km)')
        continue
    for model_name, feature_cols in [('CLR', CLR_COLS), ('CLR+GW', CLR_COLS + GW_COLS)]:
        X = sub[feature_cols].fillna(0).values
        n_splits = min(5, int(len(sub) / 4))
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
        oof = np.zeros(len(y_reg))
        for tr, te in kf.split(X):
            pipe = Pipeline([('sc', StandardScaler()), ('rr', Ridge(alpha=1.0))])
            pipe.fit(X[tr], y_reg[tr])
            oof[te] = pipe.predict(X[te])
        r2 = r2_score(y_reg, oof)
        within_rows.append({'model': model_name, 'region': region,
                             'n': len(sub), 'r2': round(r2, 4)})
    print(f'Region {region}: n={len(sub):4d}  '
          f'CLR R²={within_rows[-2]["r2"]:+.4f}  '
          f'CLR+GW R²={within_rows[-1]["r2"]:+.4f}')

within_df = pd.DataFrame(within_rows)
within_df.to_csv(DATA / 'h8_extended_regression_results.csv', index=False)

Region 0: n=1465  CLR R²=+0.7849  CLR+GW R²=+0.7999


Region 1: n=9822  CLR R²=+0.7567  CLR+GW R²=+0.7653
Region 2: n= 237  CLR R²=+0.1574  CLR+GW R²=+0.3639
Region 3: n=2108  CLR R²=+0.9243  CLR+GW R²=+0.9255
Region 4: n= 142  CLR R²=+0.7508  CLR+GW R²=-0.9394
Region 5: n= 174  CLR R²=+0.8764  CLR+GW R²=+0.6056
Region 6: n=1138  CLR R²=+0.2941  CLR+GW R²=-0.9213


Region 7: n= 390  CLR R²=-0.5106  CLR+GW R²=-0.6240


In [8]:
# H8 extended verdict
if len(within_df) == 0:
    print('No usable regions — H8 extended: UNTESTABLE')
else:
    pivot = within_df.pivot(index='region', columns='model', values='r2').round(4)
    print('\nWithin-region R² (log mine proximity regression):')
    print(pivot)

    n_usable = len(pivot)
    clr_above_thresh = (pivot.get('CLR', pd.Series(dtype=float)) > 0.05).sum()
    if 'CLR+GW' in pivot.columns and 'CLR' in pivot.columns:
        delta_r2 = (pivot['CLR+GW'] - pivot['CLR']).mean()
    else:
        delta_r2 = float('nan')

    print(f'\nUsable regions: {n_usable} (of 8)')
    print(f'Regions with CLR R² > 0.05: {clr_above_thresh}')
    print(f'Mean Δ R² (CLR+GW − CLR): {delta_r2:.4f}')

    h8_ext_supported = (clr_above_thresh >= 4) and (delta_r2 >= 0.01)
    print(f'\nH8 EXTENDED OUTCOME: {"SUPPORTED" if h8_ext_supported else "NOT SUPPORTED"}')


Within-region R² (log mine proximity regression):
model      CLR  CLR+GW
region                
0       0.7849  0.7999
1       0.7567  0.7653
2       0.1574  0.3639
3       0.9243  0.9255
4       0.7508 -0.9394
5       0.8764  0.6056
6       0.2941 -0.9213
7      -0.5106 -0.6240

Usable regions: 8 (of 8)
Regions with CLR R² > 0.05: 7
Mean Δ R² (CLR+GW − CLR): -0.3823

H8 EXTENDED OUTCOME: NOT SUPPORTED


## Section D: Summary figure

In [9]:
metals  = ['Cu', 'Zn', 'Pb', 'Ni']
targets = ['log_Cu_ppm', 'log_Zn_ppm', 'log_Pb_ppm', 'log_Ni_ppm']

fig, axs = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H))

# Left: absolute RMSE
ax = axs[0]
x = np.arange(len(metals))
w = 0.25
b0_vals   = [b0_rmse.get(t, np.nan) for t in targets]
m2_vals   = [m2_rmse.get(t, np.nan) for t in targets]
mine_vals = [mine_cv_df.loc[t, 'rmse'] for t in targets]

ax.bar(x - w, b0_vals,   width=w, label='B0',      color=PALETTE[0], edgecolor='k', lw=0.5)
ax.bar(x,     m2_vals,   width=w, label='M2',      color=PALETTE[1], edgecolor='k', lw=0.5)
ax.bar(x + w, mine_vals, width=w, label='M2_mine', color=PALETTE[2], edgecolor='k', lw=0.5)
ax.set_xticks(x)
ax.set_xticklabels(metals)
ax.set_xlabel('Target metal')
ax.set_ylabel('RMSE (log1p ppm, spatial block CV)')
ax.set_title('Spatial block CV RMSE')
ax.legend(fontsize=8)
grid_h(ax)

# Right: Δ RMSE (M2_mine − M2)
ax = axs[1]
delta = [mine_vals[i] - m2_vals[i] for i in range(len(targets))]
colors = [PALETTE[3] if d < 0 else PALETTE[4] for d in delta]
ax.bar(metals, delta, color=colors, edgecolor='k', lw=0.5)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Target metal')
ax.set_ylabel('ΔRMSE (M2_mine − M2)')
ax.set_title('Mine proximity: improvement over M2')
grid_h(ax)

fig.suptitle('NB09: Mine proximity features — spatial block CV', y=1.02)
save(fig, FIGS / 'fig_nb09_mine_proximity_cv')
print('Saved fig_nb09_mine_proximity_cv.pdf')

Saved fig_nb09_mine_proximity_cv.pdf
